# Hierarchical Split Indexing & Parent Document Retrieval

## 1. The Core Retrieval-Generation Dilemma
In dense retrieval RAG systems, document chunk size dictates a fundamental trade-off:
- **Small Chunks (e.g. 100-200 tokens)**: High semantic specificity. Dense vector embeddings accurately capture fine-grained facts without averaging out key entities across long text, resulting in high cosine similarity against short queries.
- **Large Chunks (e.g. 1000-2000 tokens)**: Rich context. Generating correct, hallucination-free LLM answers requires surrounding paragraph context, explanations, and qualifications that are lost if only a snippet is passed.

### The Parent Document Retrieval Solution
Parent Document Retrieval decouples the **retrieval index** from the **generation context**:
1. Documents are first split into large **Parent Documents** (e.g. 1000 tokens) and stored in a key-value Docstore.
2. Each parent document is recursively sub-split into small **Child Chunks** (e.g. 150 tokens) containing a metadata pointer (`parent_id`).
3. Dense vector search is performed strictly over child chunks.
4. When top-$k$ child chunks match, the system de-duplicates their `parent_id` references and fetches the full parent documents for LLM context synthesis!

In [ ]:
import numpy as np

class SimulatedParentDocumentRetriever:
    """
    Demonstrates Hierarchical Chunking and Child-to-Parent Docstore Resolution.
    """
    def __init__(self, child_chunk_size=40, parent_chunk_size=160):
        self.child_chunk_size = child_chunk_size
        self.parent_chunk_size = parent_chunk_size
        self.docstore = {} # parent_id -> parent_text
        self.child_index = [] # list of (child_id, parent_id, child_text, embedding)

    def add_document(self, doc_id, full_text):
        """Splits full document into parents, then parents into children."""
        words = full_text.split()
        
        # 1. Create Parent Documents
        for p_idx in range(0, len(words), self.parent_chunk_size):
            parent_words = words[p_idx:p_idx + self.parent_chunk_size]
            parent_text = " ".join(parent_words)
            parent_id = f"{doc_id}_p{p_idx // self.parent_chunk_size}"
            self.docstore[parent_id] = parent_text
            
            # 2. Subdivide into Child Chunks
            for c_idx in range(0, len(parent_words), self.child_chunk_size):
                child_words = parent_words[c_idx:c_idx + self.child_chunk_size]
                child_text = " ".join(child_words)
                child_id = f"{parent_id}_c{c_idx // self.child_chunk_size}"
                
                # Synthetic embedding (mean word length simulation for demo)
                dummy_emb = np.array([len(w) for w in child_words[:8]], dtype=np.float32)
                dummy_emb = np.pad(dummy_emb, (0, max(0, 8 - len(dummy_emb))))
                dummy_emb = dummy_emb / (np.linalg.norm(dummy_emb) + 1e-9)
                
                self.child_index.append({
                    "child_id": child_id,
                    "parent_id": parent_id,
                    "text": child_text,
                    "embedding": dummy_emb
                })

    def retrieve(self, query_emb, top_k_children=3):
        """Retrieves top child chunks and resolves to unique parent documents."""
        # Compute cosine similarity with all children
        scores = []
        for item in self.child_index:
            score = float(np.dot(query_emb, item["embedding"]))
            scores.append((score, item))
            
        scores.sort(key=lambda x: x[0], reverse=True)
        top_matches = scores[:top_k_children]
        
        # Resolve to unique parent documents
        resolved_parents = {}
        for score, match in top_matches:
            p_id = match["parent_id"]
            if p_id not in resolved_parents:
                resolved_parents[p_id] = {
                    "score": score,
                    "matched_child": match["text"],
                    "parent_context": self.docstore[p_id]
                }
        return resolved_parents

# Verification
retriever = SimulatedParentDocumentRetriever(child_chunk_size=10, parent_chunk_size=30)
long_document = (
    "Machine learning models require robust validation strategies to prevent overfitting. "
    "K-Fold cross-validation divides data into K subsets and trains iteratively. "
    "However, for time series data, standard K-Fold causes lookahead bias. "
    "Instead, expanding window or rolling window backtesting must be employed. "
    "In financial applications, purges and embargoes are additionally critical."
)

retriever.add_document("doc_01", long_document)
print(f"Parent Documents in Docstore: {len(retriever.docstore)}")
print(f"Indexed Child Chunks: {len(retriever.child_index)}")

dummy_query_emb = np.ones(8) / np.sqrt(8)
results = retriever.retrieve(dummy_query_emb, top_k_children=2)
for pid, info in results.items():
    print(f"
Resolved Parent [{pid}] (Score: {info['score']:.3f}):")
    print(f"  Matched Child Snippet : '{info['matched_child'][:50]}...'")
    print(f"  Full Parent Context   : '{info['parent_context']}'")

## 2. Quantitative Trade-off Matrix

| Criterion | Naive Small Chunks (100T) | Naive Large Chunks (1500T) | Parent Document Retrieval |
| :--- | :--- | :--- | :--- |
| **Retrieval Precision (Dense Cosine)** | Extremely High | Moderate / Low (Diluted) | **Extremely High (Indexed on Child)** |
| **Context Completeness for LLM** | Poor (Fragmented) | High | **High (Parent returned to LLM)** |
| **Vector DB Index Size** | High | Low | Moderate |
| **Hallucination Risk** | High (Context missing) | Moderate | **Lowest** |

## 3. High-Yield Interview Questions & Best Practices

### Q1: How does Parent Document Retrieval solve the "Lost in the Middle" attention degradation problem?
**Answer:** "Lost in the Middle" refers to LLMs attending strongly to the beginning and end of long prompts while ignoring information buried in the middle. With Parent Document Retrieval, because retrieval is matched against sharp child chunks, the system can pass only the exact parent section (e.g. 500-1000 tokens) rather than dumping an entire 10,000-token PDF into the prompt. This focuses LLM attention directly on the authoritative paragraph containing the evidence.

### Q2: What happens if multiple retrieved child chunks belong to the exact same parent document?
**Answer:** The retriever deduplicates parent IDs during the resolution step. If 3 of the top-5 retrieved children belong to `doc_01_p2`, that parent document is added to the generation context **only once**. This preserves LLM context window space and prevents repetitive token generation.